In [0]:
!pip install xgboost
dbutils.library.restartPython()

In [0]:
%matplotlib inline
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import product, combinations
from collections import Counter
from functools import reduce
from math import gcd
import xgboost as xgb
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
# from sklearn.linear_model import LogisticRegression
# from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
# from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer

In [0]:
balanced_df= pd.read_parquet('/Volumes/workspace/default/fraud_project/train_test_data/balancedFraudTrain.parquet')
test_df= pd.read_parquet('/Volumes/workspace/default/fraud_project/train_test_data/fraudTest.parquet')

In [0]:
X= balanced_df.drop(columns=['is_fraud'])
y= balanced_df['is_fraud']

cat_cols= X.select_dtypes(include='object').columns.tolist()
num_cols= X.select_dtypes(exclude='object').columns.tolist()

num_pipeline= Pipeline([
    ('scaler', RobustScaler())
])

# Linear Pipeline

In [0]:
# Works properly only with Linear Classifiers

linear_cat_pipeline= Pipeline([
    ('OHE', OneHotEncoder(handle_unknown='ignore', drop='first'))
])


linear_prep= ColumnTransformer(
    [
        ('linear_cat', linear_cat_pipeline, cat_cols),
        ('linear_num', num_pipeline, num_cols)
     ], remainder='passthrough'
)

# Tree Pipeline

In [0]:
# Works efficiently with tree & ensemble based models, & works well with linear classifiers as well

tree_cat_pipeline= Pipeline([
    ('Ordinal encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
])

tree_prep= ColumnTransformer(
    [
        ('tree_cat', tree_cat_pipeline, cat_cols),
        ('tree_num', num_pipeline, num_cols)
     ], remainder='passthrough'
)

In [0]:
X_train, X_test, y_train, y_test= train_test_split(X, y, train_size=0.8, stratify=y)
kf= KFold(n_splits=5, shuffle=True, random_state=42)

In [0]:
X_train_linear= linear_prep.fit_transform(X_train)
X_test_linear= linear_prep.transform(X_test)
X_train_tree= tree_prep.fit_transform(X_train)
X_test_tree= tree_prep.transform(X_test)

In [0]:
print(X_train_linear.shape)
print(X_train_tree.shape)

In [0]:
n= 3
weights= list(product([1,2,3], repeat=n))
len(weights)

In [0]:
weights= [
    w for w in weights 
    if max(Counter(w).values())<=2
    ]
len(weights)

In [0]:
def normalise(w):
    g= reduce(gcd,w)
    return tuple(x // g for x in w)

weights= list(set(normalise(w) for w in weights))
len(weights)

In [0]:
X_val, _, y_val, _= train_test_split(X_test_tree, y_test, train_size=0.1, shuffle=True, random_state=42)

In [0]:
# estimators= {
#     'gbt': GradientBoostingClassifier(learning_rate=0.505, loss='exponential', n_estimators=300, random_state=42),
#     'xgb': xgb.XGBClassifier(eval_metric='logloss', max_depth=15, n_estimators=100, objective='binary:logistic', reg_lambda=5, random_state=42, early_stopping_rounds=100, n_jobs=1),
#     'et': ExtraTreesClassifier(ccp_alpha=0.0001, max_features=None, n_jobs=1, random_state=42),
#     'lr': LogisticRegression(max_iter=5000, random_state=42, n_jobs=1),
    # 'rf': RandomForestClassifier(ccp_alpha=0.0001, max_depth=20, n_jobs=1, random_state=42),
    # 'dt': DecisionTreeClassifier(ccp_alpha=0.0001, max_depth=10, random_state=42)
# }

# for name, model in estimators.items():
#     if name=='xgb':
#         model.fit(X_train_tree, y_train, eval_set=[(X_val,y_val)])
#     else:
#         model.fit(X_train_tree, y_train)
#     joblib.dump(model, f"{name}_model_tree_vote") ## tree_prep preprocessed data is being used 
#     print(name)

# del estimators

In [0]:
gbt_est= joblib.load('pre-trained/tree_prep/gbt')
xgb_est= joblib.load('pre-trained/tree_prep/xgb')
et_est= joblib.load('pre-trained/tree_prep/et')
lr_est= joblib.load('pre-trained/tree_prep/lr')
rf_est= joblib.load('pre-trained/tree_prep/rf')
dt_est= joblib.load('pre-trained/tree_prep/dt')

In [0]:
estimators= [
    ('gbt', gbt_est), 
    ('xgb', xgb_est), 
    ('et', et_est), 
    ('lr', lr_est), 
    ('rf', rf_est), 
    ('dt', dt_est)
]
xgb_est.set_params(early_stopping_rounds=None)

In [0]:
combo= [list(c) for c in combinations(estimators,n)]

In [0]:
vote_pipe= Pipeline([
    ('model', VotingClassifier(estimators=[('xgb', xgb_est)]))
    ])

vote_grid= {
    'model__estimators':combo,
    'model__voting':['soft'],
    'model__weights':weights
}

vote_search= GridSearchCV(vote_pipe, param_grid=vote_grid, cv=kf, n_jobs=-1, scoring='recall')
vote_search.fit(X_train_tree, y_train)
preds= vote_search.best_estimator_.predict(X_test_tree)
print(vote_search.best_score_)
print(vote_search.best_params_)

In [0]:
# joblib.dump(vote_search.best_estimator_, 'tri_model_recall')

In [0]:
tri_model_recall= joblib.load('tri_model_recall')

In [0]:
cm= metrics.confusion_matrix(y_test, tri_model_recall.predict(X_test_tree))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
cm= metrics.confusion_matrix(test_df['is_fraud'], tri_model_recall.predict(tree_prep.transform(test_df.drop(columns=['is_fraud']))))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
joblib.dump(vote_search.best_estimator_, 'dual_model_recall')

In [0]:
four_model= joblib.load('four_model_vote_clf')

In [0]:
cm= metrics.confusion_matrix(y_test, four_model.predict(X_test_tree))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
cm= metrics.confusion_matrix(test_df['is_fraud'], four_model.predict(tree_prep.transform(test_df.drop(columns=['is_fraud']))))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
four_model.get_params()

In [0]:
cm= metrics.confusion_matrix(y_test, vote_search.best_estimator_.predict(X_test_tree))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
cm= metrics.confusion_matrix(test_df['is_fraud'], vote_search.best_estimator_.predict(tree_prep.transform(test_df.drop(columns=['is_fraud']))))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
joblib.dump(vote_search.best_estimator_,'four_model_vote_clf')

In [0]:
(77*20)+10653

In [0]:
tri_model= joblib.load('tri_model_weighted_voting_classifier_f1')
preds= tri_model.predict(X_test_tree)

In [0]:
dual_model= joblib.load('weighted_trained/dual_model_recall')
preds= dual_model.predict(X_test_tree)

In [0]:
cm= metrics.confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
cm= metrics.confusion_matrix(test_df['is_fraud'], dual_model.predict(tree_prep.transform(test_df.drop(columns=['is_fraud']))))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
9280+(87*20)

In [0]:
pipe= Pipeline([
    ('model',RandomForestClassifier())
])

param_grid= [
    # {
    #     'model':[RandomForestClassifier()],
    #     'model__max_depth':[10,15,20,None],
    #     'model__max_features':['sqrt','log2',None],
    #     'model__ccp_alpha':[1e-4,1e-2,0.2,0.5],
    #     'model__n_jobs':[-1],
    #     'model__random_state':[42]
    # },
    # {
    #     'model':[GradientBoostingClassifier()],
    #     'model__loss':['log_loss','exponential'],
    #     'model__learning_rate':np.linspace(0.01,1,5),
    #     'model__n_estimators':[100,200,300],
    #     'model__random_state':[42]
    # }#,
    {
        'model':[ExtraTreesClassifier()],
        'model__max_depth':[10,15,20,None],
        'model__max_features':['sqrt','log2',None],
        'model__ccp_alpha':[1e-4,1e-2,0.2,0.5],
        'model__n_jobs':[-1],
        'model__random_state':[42]
    }#,
    # {
    #     'model':[DecisionTreeClassifier()],
    #     'model__max_depth':[3,5,10,15,20,None],
    #     'model__max_features':['sqrt','log2',None],
    #     'model__random_state':[42],
    #     'model__ccp_alpha':[1e-4,1e-2,0.2]
    # }
    ]

grid_search= GridSearchCV(pipe, param_grid=param_grid, cv=kf, scoring='roc-auc',n_jobs=-1)
grid_search.fit(X_train_tree, y_train)
print(grid_search.best_params_)
print(grid_search.best_score_)

In [0]:
cm= metrics.confusion_matrix(y_test, grid_search.best_estimator_.predict(X_test_tree))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
cm= metrics.confusion_matrix(test_df['is_fraud'], grid_search.best_estimator_.predict(tree_prep.transform(test_df.drop(columns=['is_fraud']))))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
weights= list(product([1,2,3], repeat=6))

print(len(weights))


weights= [
    w for w in weights 
    if max(Counter(w).values())<=3
    ]

print(len(weights))

def normalise(w):
    g= reduce(gcd,w)
    return tuple(x // g for x in w)

weights= list(set(normalise(w) for w in weights))
print(len(weights))

In [0]:
vote= VotingClassifier(estimators, voting='soft')

vote_pipe= Pipeline([
    ('model',vote)
])

vote_grid= [
    {
        'model__voting':['soft', 'hard'],
        'model__weights':weights
    }
]
vote_search= GridSearchCV(vote_pipe, param_grid=vote_grid, cv=3, n_jobs=-1, scoring='recall')
vote_search.fit(X_train_tree, y_train)
vote_clf= vote_search.best_estimator_
print(vote_search.best_score_)
print(vote_search.best_params_)

In [0]:
cm= metrics.confusion_matrix(test_df['is_fraud'], vote_clf.predict(tree_prep.transform(test_df.drop(columns=['is_fraud']))))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')